# FBBP — Analyse de performance
## Chapitre 1 : Manques offensifs de l'équipe

**Objectif** : identifier les lacunes offensives du FBBP sur 5 matchs (J22 → J26) avant d'en chercher les causes.

**Matchs analysés** :
- J22 • Orléans — Dom • L (0-3)
- J23 • QRM — Ext • D (2-2)
- J24 • Rouen — Dom • W (3-1)
- J25 • Versailles — Ext • L (0-1)
- J26 • Stade Briochin — Dom • D (1-1)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

OUTPUT_DIR = '../outputs/chap1_charts'

# ── Style global ──────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f8f6',
    'axes.grid': True,
    'grid.color': '#e8e8e5',
    'grid.linewidth': 0.8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.spines.left': False,
    'axes.spines.bottom': False,
    'xtick.bottom': False,
    'ytick.left': False,
    'axes.labelcolor': '#555',
    'xtick.color': '#555',
    'ytick.color': '#555',
})

# ── Palette ───────────────────────────────────────────────
BLUE   = "#548ddb"
GRAY   = '#B4B2A9'
GREEN  = '#3B6D11'
RED    = '#A32D2D'
LGREEN = '#EAF3DE'
LRED   = '#FCEBEB'
ORANGE = '#BA7517'
BLACK  = '#333333'

RES_COLOR = {'W': GREEN, 'D': ORANGE, 'L': RED}

print('✅ Librairies chargées')

In [ ]:
df_raw = pd.read_csv('../data/01_matches_summary.csv')

# Identifier si FBBP est home ou away
def get_fbbp_stats(row):
    if row['home_team'] == 'FBBP':
        return pd.Series({
            'xg_fbbp':      row['home_xg'],
            'xg_adv':       row['away_xg'],
            'tirs_fbbp':    row['home_shots'],
            'tirs_adv':     row['away_shots'],
            'dist_tir':     row['home_avg_shot_distance'],
            'dist_tir_adv': row['away_avg_shot_distance'],
            'ppda_fbbp':    row['home_ppda'],
            'ppda_adv':     row['away_ppda'],
            'lieu':         'Dom',
            'adversaire':   row['away_team'],
            'resultat':     'W' if row['home_goals'] > row['away_goals'] else ('D' if row['home_goals'] == row['away_goals'] else 'L'),
            'score':        f"{row['home_goals']}-{row['away_goals']}",
        })
    else:
        return pd.Series({
            'xg_fbbp':      row['away_xg'],
            'xg_adv':       row['home_xg'],
            'tirs_fbbp':    row['away_shots'],
            'tirs_adv':     row['home_shots'],
            'dist_tir':     row['away_avg_shot_distance'],
            'dist_tir_adv': row['home_avg_shot_distance'],
            'ppda_fbbp':    row['away_ppda'],
            'ppda_adv':     row['home_ppda'],
            'lieu':         'Ext',
            'adversaire':   row['home_team'],
            'resultat':     'W' if row['away_goals'] > row['home_goals'] else ('D' if row['away_goals'] == row['home_goals'] else 'L'),
            'score':        f"{row['away_goals']}-{row['home_goals']}",
        })

df = df_raw.apply(get_fbbp_stats, axis=1)
df['journee'] = 'J' + df_raw['round'].astype(str).values
df = df.sort_values('journee').reset_index(drop=True)
df['xg_diff']     = df['xg_fbbp'] - df['xg_adv']
df['xg_per_shot'] = df['xg_fbbp'] / df['tirs_fbbp']
df['label'] = df['journee'] + '\n' + df['adversaire'] + '\n' + df['lieu'] + ' · ' + df['score']

# Ajout possessions depuis le dataset team
df_raw_team  = pd.read_csv('../data/02_teams_stats.csv')
df_team_fbbp = df_raw_team[df_raw_team['team'] == 'FBBP'].copy().reset_index(drop=True)
df_team_adv  = df_raw_team[df_raw_team['team'] != 'FBBP'].copy().reset_index(drop=True)

df['poss_surf_pct']     = (df_team_fbbp['possessions_to_box'] / df_team_fbbp['possessions'] * 100).round(1).values
df['poss_surf_pct_adv'] = (df_team_adv['possessions_to_box']  / df_team_adv['possessions']  * 100).round(1).values

## Graphique 1 — Domination xG par match

Le différentiel xG (xG FBBP − xG adversaire) mesure qui a créé les meilleures occasions.

FBBP ne domine le duel d'occasions que sur 2 matchs sur 5.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))

colors = [GREEN if v >= 0 else RED for v in df['xg_diff']]
bars = ax.bar(df['label'], df['xg_diff'], color=colors,
              width=0.55, edgecolor='white', linewidth=1)

# Valeurs sur barres
for bar, val in zip(bars, df['xg_diff']):
    y_pos = bar.get_height() + 0.04 if val >= 0 else bar.get_height() - 0.09
    va    = 'bottom' if val >= 0 else 'top'
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        y_pos,
        f"{val:+.2f}",
        ha='center', va=va,
        fontsize=10, fontweight='bold',
        color=GREEN if val >= 0 else RED
    )

ymin = df['xg_diff'].min()
ymax = df['xg_diff'].max()
ax.set_ylim(ymin * 1.25, ymax * 1.25)    
ax.axhline(0, color='#ccc', linewidth=1, linestyle='--')
ax.set_ylabel('xG différentiel (FBBP − Adversaire)', fontsize=11)
ax.set_title('Domination FBBP — xG différentiel par match', fontsize=13, pad=12, fontweight='bold')

patches = [mpatches.Patch(color=GREEN, label='Domination FBBP'),
           mpatches.Patch(color=RED,   label='Domination adverse')]
ax.legend(handles=patches, fontsize=10, frameon=False, loc='upper right')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/ch1_xg_diff.png',    dpi=150, bbox_inches='tight')
plt.show()

## Graphique 2 — Volume de tirs vs qualité par tir
Le xG par tir mesure la dangerosité moyenne de chaque occasion créée.


FBBP tire autant voire plus que ses adversaires, mais le xG par tir reste structurellement bas.

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 4.5))
ax2 = ax1.twinx()

x = np.arange(len(df))
w = 0.35

ax1.bar(x - w/2, df['tirs_fbbp'], w, label='Tirs FBBP',       color=BLUE,
        edgecolor='white', linewidth=1, zorder=2)
ax1.bar(x + w/2, df['tirs_adv'],  w, label='Tirs adversaire', color=GRAY,
        edgecolor='white', linewidth=1, zorder=2)

ax2.plot(x, df['xg_per_shot'], color=ORANGE, marker='o',
         linewidth=2, markersize=6, linestyle='--', label='xG/tir FBBP', zorder=3)
ax2.axhline(df['xg_per_shot'].mean(), color=ORANGE, linewidth=0.8,
            linestyle=':', alpha=0.5)

for i, val in enumerate(df['xg_per_shot']):
    ax2.text(i, val + 0.013, f'{val:.3f}', ha='center', fontsize=10,
             color=BLACK, fontweight='bold')

ax1.set_xticks(x)
ax1.set_xticklabels(df['label'], fontsize=9)
ax1.set_ylabel('Nombre de tirs', fontsize=11)
ax2.set_ylabel('Qualité des occasions FBBP (xG / tir)', fontsize=11, color=ORANGE)
ax2.tick_params(axis='y', colors=ORANGE)
ax2.set_ylim(0, 0.25)
ax2.grid(False)

ax1.set_title('Volume de tirs vs qualité des occasions — FBBP', fontsize=13, pad=12, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=10, frameon=False)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/ch1_tirs_qualite.png', dpi=150, bbox_inches='tight')
plt.show()


## Graphique 3 — Distance moyenne de tir
La distance moyenne de tir indique à quel point l'équipe arrive à se rapprocher du but.

FBBP tire en moyenne à 21.9m, bien au-delà du seuil des 16m considéré comme dangereux.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))

x = np.arange(len(df))
h = 0.35

b1 = ax.barh(x - h/2, df['dist_tir'],     h, label='FBBP',       color=BLUE, edgecolor='white', linewidth=1)
b2 = ax.barh(x + h/2, df['dist_tir_adv'], h, label='Adversaire', color=GRAY, edgecolor='white', linewidth=1)

for bar, val in zip(b1, df['dist_tir']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f} m', va='center', fontsize=9, color=BLUE, fontweight='bold')
for bar, val in zip(b2, df['dist_tir_adv']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f} m', va='center', fontsize=9, color='#888')

avg_fbbp = df['dist_tir'].mean()
avg_adv  = df['dist_tir_adv'].mean()

ax.axvline(avg_fbbp, color=BLUE, linewidth=1.2, linestyle=':', alpha=0.7)
ax.text(avg_fbbp + 0.3, -0.7, f'moy. FBBP\n{avg_fbbp:.1f}m', fontsize=9, color=BLUE, va='top')

ax.axvline(avg_adv, color='#888', linewidth=1.2, linestyle=':', alpha=0.7)
ax.text(avg_adv + 0.3, -0.7, f'moy. adv.\n{avg_adv:.1f}m', fontsize=9, color='#888', va='top')

ax.set_yticks(x)
ax.set_yticklabels(df['label'], fontsize=9)
ax.set_xlabel('Distance moyenne de tir (mètres)', fontsize=11)
ax.set_title('Distance moyenne de tir — FBBP vs adversaires', fontsize=13, pad=12, fontweight='bold')
ax.set_xlim(0, 35)
ax.invert_yaxis()
ax.legend(fontsize=10, frameon=False)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/ch1_dist_tir.png', dpi=150, bbox_inches='tight')
plt.show()

## Graphique 4 — Pénétration offensive — accès à la surface
Le % de possessions atteignant la surface adverse mesure la capacité à progresser jusqu'au but.

<hr style="border: none; border-top: 1.5px solid #ccc; margin: 8px 0;">

FBBP et ses adversaires atteignent la surface à un taux quasi identique (8.7% vs 9.2%). 
La pénétration n'est pas le problème.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))

x = np.arange(len(df))
w = 0.35

b1 = ax.bar(x - w/2, df['poss_surf_pct'],     w, label='FBBP',
            color=BLUE, edgecolor='white', linewidth=1)
b2 = ax.bar(x + w/2, df['poss_surf_pct_adv'], w, label='Adversaire',
            color=GRAY, edgecolor='white', linewidth=1)

for bar, val in zip(b1, df['poss_surf_pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{val:.0f}%', ha='center', fontsize=9, color=BLUE, fontweight='bold')
for bar, val in zip(b2, df['poss_surf_pct_adv']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{val:.0f}%', ha='center', fontsize=9, color='#888')

# Moyennes fbbp et adversaires
avg = df['poss_surf_pct'].mean()
avg_adv = df['poss_surf_pct_adv'].mean()

ax.axhline(avg, color=BLUE, linewidth=1, linestyle=':', alpha=0.6)
ax.text(4.55, avg - 0.8, f'moy. FBBP {avg:.1f}%', fontsize=9, color=BLUE, fontweight='bold')

ax.axhline(avg_adv, color='#888', linewidth=1, linestyle=':', alpha=0.6)
ax.text(4.55, avg_adv + 0.3, f'moy. adv. {avg_adv:.1f}%', fontsize=9, color='#888', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(df['label'], fontsize=9)
ax.set_ylabel('% possessions → surface adverse', fontsize=11)
ax.set_title('Pénétration offensive — accès à la surface', fontsize=13, pad=12, fontweight='bold')
ax.legend(fontsize=10, frameon=False)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/ch1_penetration.png',  dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
ref = {
    'xg':               df_team_adv['xg'].mean(),
    'shots':            df_team_adv['shots'].mean(),
    'xg_per_shot':      (df_team_adv['xg'] / df_team_adv['shots']).mean(),
    'avg_shot_distance': df_raw['away_avg_shot_distance'].where(df_raw['home_team'] == 'FBBP').fillna(df_raw['home_avg_shot_distance']).mean(),
    'poss_surf_pct':    (df_team_adv['possessions_to_box'] / df_team_adv['possessions'] * 100).mean(),
}

print("Références adversaires :")
for k, v in ref.items():
    print(f"  {k:25s} : {v:.2f}")

## Synthèse chapitre 1

| Indicateur | FBBP (moy.) | Moy. adversaires | Lecture |
|---|---|---|---|
| xG produit / match | 0.88 | 1.13 | FBBP crée moins et moins bien |
| xG / tir | 0.08 | 0.13 | Les adversaires créent des occasions 60% plus dangereuses |
| Distance moy. tir | 21.9 m | 19.4 m | FBBP tire 2.5m plus loin en moyenne |
| % possessions → surface | ~9% | ~9.2% | Taux similaire — le problème est dans la surface, pas avant |

<hr style="border: none; border-top: 1.5px solid #ccc; margin: 12px 0;">

FBBP atteint la surface adverse aussi souvent que ses adversaires (~9% des possessions), 
mais génère un xG/tir de 0.08 contre 0.13 pour ses adversaires. 
La pénétration n'est pas le problème — c'est la qualité des positions de tir qui fait défaut.

Pourquoi FBBP n’arrive-t-il pas à générer des tirs dangereux malgré des phases offensives régulières et une pénétration similaire dans la surface?


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))

x = np.arange(len(df))
colors = [GREEN if v <= df['xg_adv'].mean() else RED for v in df['xg_adv']]
bars = ax.bar(x, df['xg_adv'], color=colors, width=0.55, edgecolor='white', linewidth=1)

for bar, val in zip(bars, df['xg_adv']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f'{val:.2f}', ha='center', fontsize=10, fontweight='bold',
            color=GREEN if val <= df['xg_adv'].mean() else RED)

avg = df['xg_adv'].mean()
ax.axhline(avg, color='#888', linewidth=1.2, linestyle='--')
ax.text(4.55, avg + 0.03, f'moy. {avg:.2f}', fontsize=9, color='#888')

ax.set_xticks(x)
ax.set_xticklabels(df['label'], fontsize=9)
ax.set_ylim(0, df['xg_adv'].max() * 1.25)
ax.set_ylabel('xG concédé', fontsize=11)
ax.set_title('xG concédé par match — FBBP', fontsize=13, pad=12, fontweight='bold')

patches = [mpatches.Patch(color=GREEN, label='Sous la moyenne'),
           mpatches.Patch(color=RED,   label='Au-dessus de la moyenne')]
ax.legend(handles=patches, fontsize=10, frameon=False)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/ch1_xg_concede.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 4.5))
ax2 = ax1.twinx()

x = np.arange(len(df))
w = 0.35

df['xg_per_shot_adv'] = df['xg_adv'] / df['tirs_adv']

ax1.bar(x, df['tirs_adv'], w, label='Tirs subis', color=RED,
        edgecolor='white', linewidth=1, zorder=2)

ax2.plot(x, df['dist_tir_adv'], color=ORANGE, marker='o',
         linewidth=2, markersize=6, linestyle='--', label='Distance tir adverse', zorder=3)

for i, val in enumerate(df['dist_tir_adv']):
    ax2.text(i, val + 0.5, f'{val:.1f}m', ha='center', fontsize=9,
             color=ORANGE, fontweight='bold')

ax1.set_xticks(x)
ax1.set_xticklabels(df['label'], fontsize=9)
ax1.set_ylabel('Nombre de tirs subis', fontsize=11)
ax2.set_ylabel('Distance moyenne tir adverse (m)', fontsize=11, color=ORANGE)
ax2.tick_params(axis='y', colors=ORANGE)
ax2.grid(False)

ax1.set_title('Tirs subis vs distance de frappe adverse', fontsize=13, pad=12, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=10, frameon=False)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/ch1_tirs_subis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for _, row in df.iterrows():
    c = RES_COLOR[row['resultat']]
    ax.scatter(row['ppda_fbbp'], row['xg_adv'], color=c, s=120,
               edgecolors='white', linewidths=1.2, zorder=3)
    ax.annotate(row['adversaire'],
                (row['ppda_fbbp'], row['xg_adv']),
                textcoords="offset points", xytext=(8, 4),
                fontsize=9, color='#555')

avg_ppda = df['ppda_fbbp'].mean()
avg_xg   = df['xg_adv'].mean()
ax.axvline(avg_ppda, color='#ccc', linewidth=1, linestyle='--')
ax.axhline(avg_xg,   color='#ccc', linewidth=1, linestyle='--')

ax.set_xlabel('PPDA FBBP  ←  bas = pressing fort', fontsize=11)
ax.set_ylabel('xG concédé', fontsize=11)
ax.set_title('Pressing FBBP vs xG concédé', fontsize=13, pad=12, fontweight='bold')
ax.invert_xaxis()

# Annotations quadrants
ax.text(0.02, 0.97, 'Pressing fort\n+ peu concédé', transform=ax.transAxes,
        fontsize=8, color=GREEN, va='top')
ax.text(0.75, 0.97, 'Pressing faible\n+ beaucoup concédé', transform=ax.transAxes,
        fontsize=8, color=RED, va='top')

patches = [mpatches.Patch(color=c, label=l)
           for l, c in [('Victoire', GREEN), ('Nul', ORANGE), ('Défaite', RED)]]
ax.legend(handles=patches, fontsize=10, frameon=False)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/ch1_ppda_xg_concede.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))

x = np.arange(len(df))
w = 0.35

rec_high_fbbp = df_team_fbbp['recoveries_high'].values
rec_high_adv  = df_team_adv['recoveries_high'].values

b1 = ax.bar(x - w/2, rec_high_fbbp, w, label='FBBP',       color=BLUE, edgecolor='white', linewidth=1)
b2 = ax.bar(x + w/2, rec_high_adv,  w, label='Adversaire', color=GRAY, edgecolor='white', linewidth=1)

for bar, val in zip(b1, rec_high_fbbp):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(val), ha='center', fontsize=10, color=BLUE, fontweight='bold')
for bar, val in zip(b2, rec_high_adv):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(val), ha='center', fontsize=10, color='#888')

ax.set_xticks(x)
ax.set_xticklabels(df['label'], fontsize=9)
ax.set_ylabel('Récupérations hautes', fontsize=11)
ax.set_title('Récupérations hautes — FBBP vs adversaires', fontsize=13, pad=12, fontweight='bold')
ax.legend(fontsize=10, frameon=False)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/ch1_rec_hautes.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

ecarts = {
    'xG/tir (qualité occasions)': (0.10 - 0.04) / 0.04 * 100,
    'xG concédé':                 (1.67 - 0.76) / 0.76 * 100,
    'Distance de tir':            (24.95 - 19.97) / 19.97 * 100,
    'PPDA (pressing)':            (11.25 - 6.80) / 6.80 * 100,
}

labels = list(ecarts.keys())
vals   = list(ecarts.values())

bars = ax.barh(labels, vals, color=RED, edgecolor='white',
               linewidth=1, height=0.5)

for bar, val in zip(bars, vals):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2,
            f'+{val:.0f}%', va='center', fontsize=10,
            color=RED, fontweight='bold')

ax.axvline(0, color='#ccc', linewidth=1)
ax.set_xlabel('Dégradation en défaite (% d\'écart vs W/D)', fontsize=11)
ax.set_title('Qu\'est-ce qui change le plus en défaite ?', 
             fontsize=13, pad=12, fontweight='bold')
ax.set_xlim(0, max(vals) * 1.25)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/ch2_ecarts_wd_vs_l.png', dpi=150, bbox_inches='tight')
plt.show()